# Agent Tools: 擴展 AI 的能力邊界

## 目標

- 理解 Agent 為什麼需要 Tools
- 使用 Google Search 作為外部工具
- 建立自訂 Function Tools
- 將 Agent 作為另一個 Agent 的工具使用

---

## 環境設定

### 安裝必要套件

In [ ]:
# 安裝 Google ADK
!pip install -q --upgrade google-adk

### 設定 API 金鑰

**取得 API 金鑰的步驟：**

1. 前往 [Google AI Studio](https://aistudio.google.com/app/apikey)
2. 建立一個新的 API 金鑰
3. 在 Colab 左側選單中，點擊 🔑 圖示（Secrets）
4. 新增一個名為 `GOOGLE_API_KEY` 的 secret
5. 貼上您的 API 金鑰並儲存
6. 啟用該 secret 的存取權限

In [ ]:
import os
from google.colab import userdata

# 從 Colab Secrets 取得 API 金鑰
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ API 金鑰設定完成")
except Exception as e:
    print(f"❌ 錯誤：請確認您已在 Colab Secrets 中新增 'GOOGLE_API_KEY'")
    print(f"詳細資訊：{e}")

### 匯入必要模組

In [ ]:
from google.genai import types
from typing import Dict, Any
import json

print("✅ 模組匯入成功")

---

## Part 1: 為什麼 Agent 需要 Tools？

**沒有 Tools 的限制**

讓我們先建立一個沒有任何工具的 Agent，測試它的能力邊界。

In [ ]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner

# 建立一個沒有工具的 Agent
agent_no_tools = Agent(
    name="no_tools_agent",
    model=Gemini(model="gemini-flash-latest"),
    instruction="你是一個友善的助理，請用繁體中文回答。"
)

runner_no_tools = InMemoryRunner(agent=agent_no_tools)
print("✅ 無工具的 Agent 建立完成")

In [ ]:
# 測試：詢問需要即時資訊的問題
response = await runner_no_tools.run_debug(
    "今天台北的天氣如何？溫度幾度？"
)

In [ ]:
# 測試：需要計算的問題
response = await runner_no_tools.run_debug(
    "請幫我計算 123456 * 789012 = ?"
)

**💡 觀察**

沒有工具的 Agent 面臨這些限制：

- ❌ 無法取得即時資訊（天氣、新聞、股價）
- ❌ 大數字計算容易出錯
- ❌ 無法執行實際操作（發送郵件、建立檔案）
- ❌ 知識受限於訓練時間點

**這就是為什麼我們需要 Tools！**

---

## Part 2: 使用 Google Search 作為工具

### 2.1 建立具有 Google Search 能力的 Agent

In [ ]:
from google.adk.tools import google_search

# 建立具有 Google Search 工具的 Agent
agent_with_search = Agent(
    name="search_agent",
    model=Gemini(model="gemini-flash-latest"),
    instruction="""你是一個能夠搜尋網路資訊的助理。
    當使用者詢問即時資訊或你不確定的資訊時，請使用 google_search 工具來搜尋。""",
    tools=[google_search]
)

runner_with_search = InMemoryRunner(agent=agent_with_search)
print("✅ 具有搜尋能力的 Agent 建立完成")

### 2.2 測試 Google Search Tool

In [ ]:
# 測試：詢問即時資訊
response = await runner_with_search.run_debug(
    "LOL 2025 世界冠軍是哪個隊伍？"
)

**💡 觀察**

現在 Agent 可以：

- ✅ 取得即時資訊
- ✅ 查詢最新新聞和事件
- ✅ 突破知識截止日期限制
- ✅ 提供更準確的答案

---

## Part 3: 自訂 Function Tool

### 3.1 建立自訂工具

讓我們建立工具函式，展示如何將任何 Python 函式變成 Agent 工具。

In [ ]:
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """
    貨幣轉換（使用固定匯率作為示範）。

    Args:
        amount: 金額
        from_currency: 來源貨幣（USD, TWD, JPY, EUR）
        to_currency: 目標貨幣（USD, TWD, JPY, EUR）

    Returns:
        轉換後的金額
    """
    # 示範用固定匯率（實際應用應使用即時匯率 API）
    rates = {
        "USD": 1.0,
        "TWD": 31.5,
        "JPY": 149.5,
        "EUR": 0.92
    }

    try:
        # 先轉換成 USD，再轉換成目標貨幣
        amount_in_usd = amount / rates[from_currency]
        result = amount_in_usd * rates[to_currency]
        return f"{amount} {from_currency} = {result:.2f} {to_currency}"
    except KeyError:
        return f"不支援的貨幣：{from_currency} 或 {to_currency}"
    except Exception as e:
        return f"轉換錯誤：{str(e)}"

def print_function_calls(response):
    """
    Helper Function for Tracking
    """
    events = response if isinstance(response, list) else [response]

    print("\n" + "-" * 40)
    print("🔧 Function Calls:")
    print("-" * 40)

    found_calls = False
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    fc = part.function_call
                    found_calls = True
                    print(f"  Function: {fc.name}")
                    print(f"  Args: {fc.args}")
                    print(f"  ID: {fc.id}")
                    print("-" * 40)

    if not found_calls:
        print("  (沒有 function call)")

print("✅ 自訂工具定義完成")

### 3.2 建立具有自定義工具的 Agent

In [ ]:
# 建立具有自定義工具的 Agent
converter = Agent(
    name="currency_conversion_agent",
    model=Gemini(model="gemini-flash-latest"),
    instruction="""你是一個負責轉換貨幣的助理。
    根據使用者的需求，選擇適當的工具來協助。""",
    tools=[convert_currency]
)

runner_with_conversion = InMemoryRunner(agent=converter)
print("✅ 具有多種工具的 Agent 建立完成")

### 3.3 測試自訂工具

In [ ]:
# 測試貨幣轉換工具
response = await runner_with_conversion.run_debug(
    "如果我有 1000 美金，可以換成多少台幣和日幣？"
)
print_function_calls(response)

**💡 觀察**

自訂工具的優勢：

- ✅ 任何 Python 函式都可以成為工具
- ✅ Agent 會根據需求自動選擇適當的工具
- ✅ 可以組合使用多個工具完成複雜任務
- ✅ 計算結果更準確、可靠

---

## Part 4: 將 Agent 作為工具

### 4.1 概念介紹

Agent 不僅可以使用工具，**Agent 本身也可以成為另一個 Agent 的工具**！

這種模式適合：
- 專業化分工（翻譯專家、數學專家、程式專家）
- 複雜任務分解
- 建立 Agent 階層架構


### 4.2 建立專業 Agents

In [ ]:
from google.adk.code_executors import BuiltInCodeExecutor

# 建立翻譯專家 Agent
translator_agent = Agent(
    name="translator",
    model=Gemini(model="gemini-flash-latest"),
    instruction="""你是一個專業的翻譯專家。
    - 擅長英文、日文、韓文與繁體中文之間的翻譯
    - 提供準確、自然的翻譯結果
    - 保留原文的語氣和含義"""
)

# 建立數學專家 Agent
math_agent = Agent(
    name="mathematician",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""你是一個數學專家。
    - 擅長解決各種數學問題
    - 提供詳細的解題步驟
    - 使用清晰的數學符號和說明""",
    code_executor=BuiltInCodeExecutor()  # 數學專家配備計算工具
)

# 建立程式專家 Agent
programmer_agent = Agent(
    name="programmer",
    model=Gemini(model="gemini-flash-latest"),
    instruction="""你是一個 Python 程式專家。
    - 擅長編寫清晰、高效的 Python 程式碼
    - 提供完整的程式碼範例和說明"""
)

print("✅ 專業 Agents 建立完成")

### 4.3 建立協調者 Agent

現在我們建立一個「協調者 Agent」，它可以根據任務需求，呼叫不同的專業 Agent 來協助。

In [ ]:
from google.adk.tools import AgentTool

# 建立協調者 Agent
coordinator_agent = Agent(
    name="coordinator",
    model=Gemini(model="gemini-flash-latest"),
    instruction="""你是一個專案協調者，負責協調不同領域的專家來完成任務。

    你可以調用以下專家：
    - translator: 翻譯專家
    - mathematician: 數學專家
    - programmer: 程式專家

    根據使用者的需求，選擇適當的專家來協助完成任務。""",
    tools=[
        AgentTool(agent=translator_agent),
        AgentTool(agent=math_agent),
        AgentTool(agent=programmer_agent)
    ]
)

runner_coordinator = InMemoryRunner(
    agent=coordinator_agent,
)
print("✅ 協調者 Agent 建立完成")

### 4.4 測試 Agent 作為工具

In [ ]:
# 測試：需要翻譯專家
response = await runner_coordinator.run_debug(
    "請幫我把『人工智慧正在改變世界』翻譯成英文和日文。"
)
print_function_calls(response)

In [ ]:
# 測試：需要數學專家
response = await runner_coordinator.run_debug(
    "請幫我計算 123456 * 789012 = ?"
)
print_function_calls(response)

In [ ]:
# 測試：需要程式專家
response = await runner_coordinator.run_debug(
    "請幫我寫一個 Python 函式，計算費氏數列的第 n 項。"
)
print_function_calls(response)

In [ ]:
# 測試：需要多個專家協作
response = await runner_coordinator.run_debug(
    """請協助我完成以下任務：
    1. 先用數學專家計算 1+2+3+...+100 的總和
    2. 然後請程式專家寫一個 Python 函式來驗證這個結果
    3. 最後請翻譯專家把這個數學公式翻譯成英文
    """
)
print_function_calls(response)

**💡 觀察**

Agent 作為工具的優勢：

- ✅ 專業化分工，每個 Agent 專注於自己的領域
- ✅ 協調者 Agent 自動選擇適當的專家
- ✅ 可以串聯多個 Agent 完成複雜任務
- ✅ 更好的可維護性和擴展性

這就是多 Agent 系統的基礎！

---

## Part 5: 工具設計的最佳實踐

### 5.1 為什麼 Docstring 很重要？

Agent 需要理解工具的功能和使用方式，**Docstring 就是工具的使用說明書**。

In [ ]:
# 不良範例：沒有 Docstring
def bad_tool(x, y):
    return x * y

# 良好範例：完整的 Docstring
def good_tool(x: float, y: float) -> float:
    """
    計算兩個數字的乘積。

    這個工具用於精確計算數學乘法，適合需要高精度計算的場景。

    Args:
        x: 第一個數字（支援整數和浮點數）
        y: 第二個數字（支援整數和浮點數）

    Returns:
        兩個數字的乘積

    Examples:
        >>> good_tool(3, 4)
        12.0
        >>> good_tool(2.5, 4)
        10.0
    """
    return x * y

### 5.2 最佳實踐總結

**1. 完整的 Docstring**
```python
def my_tool(param1: str, param2: int) -> str:
    """
    工具的簡短描述（這很重要！Agent 會讀這個）
    
    更詳細的說明（選填）
    
    Args:
        param1: 參數說明
        param2: 參數說明
    
    Returns:
        返回值說明
    """
```

**2. 類型提示（Type Hints）**
- 幫助 Agent 理解參數類型
- 提高程式碼可讀性
- 減少錯誤

**3. 結構化返回**
- 返回清晰的結果
- 包含必要的上下文資訊
- 錯誤處理

**4. 單一職責**
- 每個工具專注於一件事
- 避免過於複雜的工具
- 易於測試和維護

---

## 總結

### 🎉 恭喜！你已完成 Agent Tools

在這個 Lab 中了解到：

**✅Tools 的重要性**
   - 理解為什麼 Agent 需要工具
   - 體驗沒有工具的限制

**✅Google Search Tool**
   - 整合外部搜尋 API
   - 取得即時資訊
   - 突破知識截止日期限制

**✅自訂 Function Tools**
   - 任何 Python 函式都可以成為工具
   - 建立多個專業化工具
   - Agent 自動選擇適當的工具

**✅Agent 作為工具**
   - 建立專業化的 Agent
   - 使用 AgentTool 包裝
   - 協調者模式實現多 Agent 協作

**✅工具設計最佳實踐**
   - Docstring 的重要性
   - 類型提示
   - 結構化返回
   - 單一職責原則